Two sources:
1. `src/dictionary_collection/mturk/analysis`
2. `src/data_term_gold/processed`

In [1]:
import pandas as pd
import json
data_6060 = pd.read_csv("/home/jiaruil5/multilingual/multilingual-model-card/src/dictionary_collection/mturk/analysis/annotation_results_6060/Japanese_validated.csv").rename(columns={"validated_translation": "validated_translation_old", "gold": "validated_translation"}, inplace=False).drop_duplicates(subset='word', inplace=False)
data_mturk = pd.read_csv("/home/jiaruil5/multilingual/multilingual-model-card/src/dictionary_collection/mturk/analysis/annotation_results_crawled/Japanese_validated.csv").drop_duplicates(subset='word', inplace=False)

In [2]:
concat_df = pd.concat([data_6060[['word', 'validated_translation']], data_mturk[['word', 'validated_translation']]], axis=0, ignore_index=True)
concat_df['word_lower'] = concat_df['word'].str.lower()
concat_df

,word,validated_translation,word_lower
0,AI,AI,ai
1,API,API,api
2,AQA,AQA,aqa
3,ARENA,ARENA,arena
4,Additionally,さらに,additionally
...,...,...,...
4985,zero-shot prompting,ゼロショットプロンプティング,zero-shot prompting
4986,zero-shot reasoning,ゼロショット推論,zero-shot reasoning
4987,zero-shot setting,ゼロショット設定,zero-shot setting
4988,zero-shot transfer,ゼロショット転送,zero-shot transfer


In [3]:
ref_df = pd.concat([pd.read_csv("/home/jiaruil5/multilingual/multilingual-model-card/src/dictionary_collection/mturk/analysis/annotation_final/Chinese.csv"), pd.read_csv("/home/jiaruil5/multilingual/multilingual-model-card/src/dictionary_collection/mturk/analysis/annotation_final/Arabic.csv"), pd.read_csv("/home/jiaruil5/multilingual/multilingual-model-card/src/dictionary_collection/mturk/analysis/annotation_final/Russian.csv")], axis=0).drop_duplicates(subset='word', inplace=False)
ref_df['word_lower'] = ref_df['word'].str.lower()

In [4]:
merged_df = concat_df
merged_df = merged_df[['word', 'validated_translation']].drop_duplicates(subset='word', inplace=False).sort_values(by='word')
merged_df['word_lower'] = merged_df['word'].str.lower()

merged = merged_df.merge(ref_df[['word', 'word_lower']], on='word_lower', how='left', suffixes=('', '_ref'))

# Update 'word' in merged_df with the 'word' from ref_df where it matches
merged['word'] = merged['word_ref'].combine_first(merged['word'])

# Drop the temporary 'word_ref' column used for merging
merged = merged.drop(columns=['word_ref'])
merged

,word,validated_translation,word_lower
0,10-fold cross validation,10分割交差検証,10-fold cross validation
1,1D convolution,1次元畳み込み,1d convolution
2,2 norm,2ノルム,2 norm
3,2D convolution,2次元畳み込み,2d convolution
4,2D image,2次元画像,2d image
...,...,...,...
4995,zero-shot prompting,ゼロショットプロンプティング,zero-shot prompting
4996,zero-shot reasoning,ゼロショット推論,zero-shot reasoning
4997,zero-shot setting,ゼロショット設定,zero-shot setting
4998,zero-shot transfer,ゼロショット転送,zero-shot transfer


In [13]:
merged[merged.duplicated(subset='word_lower')]

,word,validated_translation,word_lower
182,corpora,コーパス,corpora
193,dataset,データセット,dataset
549,morphology,形態論,morphology
691,seq2seq,シーケンス 2 シーケンス,seq2seq
793,word2vec,Word2Vec,word2vec
875,algorithm,アルゴリズム,algorithm
958,attention,アテンション,attention
1250,classifier,分類器,classifier
1316,compositionality,構成性,compositionality
1494,Corpora,コーパス,corpora


In [5]:
merged = merged.drop_duplicates(subset='word_lower', inplace=False)[['word', 'validated_translation']].sort_values(by='word')
merged.to_csv("/home/jiaruil5/multilingual/multilingual-model-card/src/dictionary_collection/mturk/analysis/annotation_final/Japanese.csv")